# Simulador do TCC: Do desejo ao resultado

Este simulador responde a duas perguntas de negócio do trabalho:

1. **A plataforma percebe a tempo quando um projeto de IA começa a dar menos resultado do que prometeu?** (hipótese H1)
2. **Vale a pena abrir a empresa que vende essa plataforma, e o que mais pesa nessa decisão?** (hipóteses H2 e H3)

**Como usar:** menu **Ambiente de execução › Executar tudo** (Ctrl+F9). Cada resposta aparece abaixo das células 2 e 3. Mova os controles para testar outros cenários; a célula recalcula sozinha (se não recalcular, clique no ▶ da célula).

Os valores iniciais são os do TCC. Para voltar a eles, reabra o arquivo.

In [ ]:
#@title 1. Preparar o simulador (execute uma vez) { display-mode: "form" }
codigo = {
    'simulador.py': r'''"""Simulador do TCC (versão para o navegador, só com NumPy).

Reproduz os scripts h1_motor_alertas.py e h2_h3_monte_carlo.py com as mesmas
sementes e os mesmos sorteios, de forma vetorizada, para rodar no Pyodide.
"""
import math
import numpy as np


# ---------------------------------------------------------------- utilidades
def _postos(x):
    ordem = np.argsort(x, kind="mergesort")
    r = np.empty(len(x), dtype=float)
    r[ordem] = np.arange(1, len(x) + 1)
    # média dos postos em empates
    xs = x[ordem]
    i = 0
    while i < len(xs):
        j = i
        while j + 1 < len(xs) and xs[j + 1] == xs[i]:
            j += 1
        if j > i:
            r[ordem[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    return r


def spearman(a, b):
    ra, rb = _postos(np.asarray(a, float)), _postos(np.asarray(b, float))
    return float(np.corrcoef(ra, rb)[0, 1])


def mann_whitney_menor(x, y):
    """U de x e p unilateral (x < y), aproximação normal com correção de empates."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n1, n2 = len(x), len(y)
    r = _postos(np.concatenate([x, y]))
    u1 = r[:n1].sum() - n1 * (n1 + 1) / 2
    _, cont = np.unique(np.concatenate([x, y]), return_counts=True)
    n = n1 + n2
    var = n1 * n2 / 12 * ((n + 1) - (cont ** 3 - cont).sum() / (n * (n - 1)))
    z = (u1 - n1 * n2 / 2 + 0.5) / math.sqrt(var)
    p = 0.5 * math.erfc(-z / math.sqrt(2))
    return float(u1), float(p)


def histograma(valores, bordas):
    c, _ = np.histogram(valores, bins=bordas)
    return c.tolist()


# ---------------------------------------------------------------- H1
def rodar_h1(p_evento=0.40, limiar=0.85, janela=4, consecutivas=2, ruido=0.15,
             queda_min=0.20, queda_max=0.50, n=1000, semanas=52, semente=42):
    rng = np.random.default_rng(semente)
    prometido = rng.lognormal(mean=np.log(20_000), sigma=0.6, size=n)
    nivel = rng.uniform(0.95, 1.10, n)
    evento = rng.random(n) < p_evento
    semana_evento = np.where(evento, rng.integers(5, 41, n), semanas + 1)
    queda = np.where(evento, rng.uniform(queda_min, queda_max, n), 0.0)
    sem = np.arange(1, semanas + 1)
    fator = np.where(sem[None, :] >= semana_evento[:, None], 1 - queda[:, None], 1.0)
    razao = np.clip(nivel[:, None] * fator + rng.normal(0, ruido, (n, semanas)), 0, None)

    # revisão trimestral
    tri = np.full(n, -1)
    for fim in (13, 26, 39, 52):
        abaixo = (razao[:, fim - 13:fim].mean(axis=1) < limiar) & (tri < 0)
        tri[abaixo] = fim

    # monitoramento contínuo: média móvel e semanas consecutivas abaixo do limiar
    cs = np.cumsum(np.concatenate([np.zeros((n, 1)), razao], axis=1), axis=1)
    media = np.full((n, semanas), np.inf)
    media[:, janela - 1:] = (cs[:, janela:] - cs[:, :-janela]) / janela
    cont = np.zeros(n, int)
    con = np.full(n, -1)
    for t in range(semanas):
        cont = np.where(media[:, t] < limiar, cont + 1, 0)
        novo = (cont >= consecutivas) & (con < 0)
        con[novo] = t + 1

    def avaliar(s):
        tem = s >= 0
        detectou = evento & tem & (s >= semana_evento)
        falso_sem_evento = (~evento) & tem
        fim_perda = np.where(detectou, s, semanas)
        perda = np.where(evento, prometido * nivel * queda * np.maximum(0, fim_perda - semana_evento + 1), 0.0)
        atraso = (s - semana_evento)[detectou]
        return dict(taxa_deteccao=float(detectou[evento].mean()),
                    atraso_mediano=float(np.median(atraso)) if len(atraso) else None,
                    atraso_medio=float(atraso.mean()) if len(atraso) else None,
                    falso_alarme=float(falso_sem_evento[~evento].mean()),
                    perda_mi=float(perda.sum() / 1e6)), atraso

    r_tri, a_tri = avaliar(tri)
    r_con, a_con = avaliar(con)
    u, p = mann_whitney_menor(a_con, a_tri)
    reducao = 1 - r_con["atraso_mediano"] / r_tri["atraso_mediano"]
    bordas = np.arange(0, 41, 1)
    return dict(
        n=n, com_queda=int(evento.sum()),
        trimestral=r_tri, continuo=r_con,
        reducao=float(reducao), u=u, p=p,
        suportada=bool(reducao >= 0.5 and r_con["falso_alarme"] <= 0.10 and p < 0.05),
        hist=dict(bordas=bordas.tolist(), trimestral=histograma(a_tri, bordas),
                  continuo=histograma(a_con, bordas)),
    )


# ---------------------------------------------------------------- modelo financeiro
NOVOS = np.array([6, 8, 8, 10, 10], float)
ASSINATURA = np.array([0.180, 0.220, 0.230, 0.240, 0.250])
AGENTES_BASE = np.array([0, 4, 8, 12, 16], float)
FIXOS = np.array([2.2, 3.4, 4.3, 5.4, 6.4])
DIAG, AGENTE, DESC1, TRIB, TAXA, CRESC = 0.020, 0.050, 0.75, 0.14, 0.25, 0.05
BASE = dict(mult_clientes=1.0, mult_preco=1.0, aceite=0.80, custo_var=0.06, mult_fixos=1.0, churn=0.0)


def projetar(mult_clientes, mult_preco, aceite, custo_var, mult_fixos, churn):
    """Vetorizado: cada argumento pode ser escalar ou vetor de cenários."""
    mc, mp, ac, cv, mf, ch = (np.atleast_1d(np.asarray(v, float)) for v in
                              (mult_clientes, mult_preco, aceite, custo_var, mult_fixos, churn))
    k = max(len(mc), len(mp), len(ac), len(cv), len(mf), len(ch))
    mc, mp, ac, cv, mf, ch = (np.broadcast_to(v, (k,)) for v in (mc, mp, ac, cv, mf, ch))
    novos = mc[:, None] * NOVOS[None, :]
    fim = np.zeros((k, 5))
    ant = np.zeros(k)
    for t in range(5):
        fim[:, t] = ant * (1 - ch) + novos[:, t]
        ant = fim[:, t]
    inicio = np.concatenate([np.zeros((k, 1)), fim[:, :-1]], axis=1)
    medio = (inicio + fim) / 2
    medio[:, 0] = fim[:, 0] / 4
    assin = medio * ASSINATURA[None, :] * mp[:, None]
    proj_novo = (DIAG + AGENTE * ac) * mp
    fator_ano = np.array([DESC1, 1, 1, 1, 1])
    projetos = novos * proj_novo[:, None] * fator_ano[None, :] + \
        AGENTES_BASE[None, :] * mc[:, None] * (AGENTE * ac * mp)[:, None]
    receita = assin + projetos
    resultado = receita * (1 - TRIB - cv)[:, None] - FIXOS[None, :] * mf[:, None]
    desc = (1 + TAXA) ** np.arange(1, 6)
    vp_anos = (resultado / desc).sum(axis=1)
    vp_term = resultado[:, -1] * (1 + CRESC) / (TAXA - CRESC) / desc[-1]
    acum = np.cumsum(resultado, axis=1)
    pos = resultado > 0
    equil = np.where(pos.any(axis=1), pos.argmax(axis=1) + 1, 99)
    return dict(assinaturas=assin, projetos=projetos, receita=receita, resultado=resultado,
                vp_anos=vp_anos, vp_terminal=vp_term, vpl=vp_anos + vp_term,
                caixa_min=acum.min(axis=1), equilibrio=equil)


NOMES = {"mult_clientes": "Ritmo de aquisição de clientes", "mult_preco": "Preço praticado",
         "aceite": "Taxa de aceite dos agentes", "custo_var": "Custo variável de IA e voz",
         "mult_fixos": "Custos fixos", "churn": "Rotatividade de clientes"}
# extremo desfavorável é o máximo para custos e rotatividade
DESFAVORAVEL_NO_MAXIMO = {"custo_var", "mult_fixos", "churn"}


def rodar_h2(faixas=None, n=10_000, meta=0.70, semente=42):
    f = {"mult_clientes": (0.50, 1.00, 1.20), "mult_preco": (0.80, 1.00, 1.10),
         "aceite": (0.60, None, 0.90), "custo_var": (0.04, 0.06, 0.12),
         "mult_fixos": (0.90, 1.00, 1.15), "churn": (0.00, 0.05, 0.20)}
    if faixas:
        f.update({k: tuple(v) for k, v in faixas.items()})
    rng = np.random.default_rng(semente)

    def tri(k):
        a, m, b = f[k]
        m = min(max(m, a), b)
        return np.full(n, a) if b <= a else rng.triangular(a, m, b, n)

    pr = {}
    pr["mult_clientes"] = tri("mult_clientes")
    pr["mult_preco"] = tri("mult_preco")
    a, _, b = f["aceite"]
    pr["aceite"] = rng.uniform(a, b, n)
    pr["custo_var"] = tri("custo_var")
    pr["mult_fixos"] = tri("mult_fixos")
    pr["churn"] = tri("churn")

    s = projetar(**pr)
    vpl = s["vpl"]
    p_pos = float((vpl > 0).mean())
    erro = 1.96 * math.sqrt(p_pos * (1 - p_pos) / n)
    p10, p50, p90 = (float(x) for x in np.percentile(vpl, [10, 50, 90]))

    rho = {k: spearman(pr[k], vpl) for k in NOMES}
    base = projetar(**BASE)
    vpl_base = float(base["vpl"][0])
    tornado = []
    for k in NOMES:
        lo, hi = f[k][0], f[k][2]
        ruim, bom = (hi, lo) if k in DESFAVORAVEL_NO_MAXIMO else (lo, hi)
        vr = float(projetar(**{**BASE, k: ruim})["vpl"][0])
        vb = float(projetar(**{**BASE, k: bom})["vpl"][0])
        tornado.append(dict(nome=NOMES[k], pior=vr, melhor=vb, rho=rho[k]))
    tornado.sort(key=lambda t: -abs(t["melhor"] - t["pior"]))
    maior = max(rho, key=lambda k: abs(rho[k]))

    cond = vpl[pr["mult_clientes"] >= 1.0]
    lo_e, hi_e = float(vpl.min()), float(vpl.max())
    bordas = np.linspace(lo_e, hi_e, 61) if hi_e > lo_e else np.array([lo_e - 1, lo_e + 1])
    return dict(
        n=n, p_pos=p_pos, ic=[max(0.0, p_pos - erro), min(1.0, p_pos + erro)],
        p10=p10, p50=p50, p90=p90, media=float(vpl.mean()),
        caixa_p50=float(-np.percentile(s["caixa_min"], 50)),
        caixa_p90=float(-np.percentile(s["caixa_min"], 10)),
        equilibrio_5a=float((s["equilibrio"] <= 5).mean()),
        p_pos_ritmo_pleno=float((cond > 0).mean()) if len(cond) else None,
        h2_suportada=bool(p_pos - erro >= meta), meta=meta,
        h3_suportada=bool(maior == "mult_clientes"), mais_influente=NOMES[maior],
        tornado=tornado, vpl_base=vpl_base,
        base=dict(receita=base["receita"][0].round(2).tolist(),
                  resultado=base["resultado"][0].round(2).tolist(),
                  caixa=float(-base["caixa_min"][0]), equilibrio=int(base["equilibrio"][0])),
        hist=dict(bordas=bordas.tolist(),
                  negativo=histograma(vpl[vpl <= 0], bordas),
                  positivo=histograma(vpl[vpl > 0], bordas)),
    )
''',
    'relatorio.py': r'''"""Relatório das hipóteses do TCC em linguagem de negócio (HTML + matplotlib)."""
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, HTML
import simulador as sim

AZUL, LARANJA, TINTA, CINZA, GRADE = "#2a78d6", "#eb6834", "#14212b", "#55606b", "#d5dbe1"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRADE, "axes.labelcolor": CINZA,
                     "xtick.color": CINZA, "ytick.color": CINZA, "figure.dpi": 110})

CSS = """
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@600&family=IBM+Plex+Sans:wght@400;600&family=Newsreader:opsz,wght@6..72,600&display=swap">
<style>
.tcc{font-family:'IBM Plex Sans',system-ui,sans-serif;color:#14212b;background:#fff;border:1px solid #d5dbe1;
     border-radius:10px;padding:20px 22px;max-width:980px;line-height:1.5}
.tcc h3{font-family:'Newsreader',Georgia,serif;font-weight:600;font-size:22px;margin:0 0 6px}
.tcc h4{font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:#55606b;margin:18px 0 6px}
.tcc p{margin:0 0 8px;max-width:72ch}
.tcc .pergunta{font-size:16px;margin-bottom:10px}
.tcc .resposta{border-left:4px solid var(--cor);background:var(--fundo);padding:10px 14px;border-radius:0 8px 8px 0;margin:10px 0 4px}
.tcc .resposta b{display:block;font-size:15px}
.tcc .ok{--cor:#1d6b43;--fundo:#e8f5ee}.tcc .nao{--cor:#9b3b16;--fundo:#fcece5}.tcc .parcial{--cor:#8a6100;--fundo:#fdf4dc}
.tcc .grade{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:10px;margin:6px 0 4px}
.tcc .kpi{border:1px solid #e3e8ec;border-radius:8px;padding:10px 12px;background:#f7f9fa}
.tcc .kpi b{display:block;font-family:'IBM Plex Mono',ui-monospace,monospace;font-size:22px;font-weight:600;
     font-variant-numeric:tabular-nums}
.tcc .kpi span{font-size:13px;color:#3d4852}
.tcc table{border-collapse:collapse;width:100%;font-size:13px;font-variant-numeric:tabular-nums;margin-top:4px}
.tcc th,.tcc td{padding:6px 8px;border-bottom:1px solid #e3e8ec;text-align:right;vertical-align:top}
.tcc th:first-child,.tcc td:first-child{text-align:left}
.tcc th{color:#55606b;font-weight:600;font-size:12px}
.tcc ul{margin:0 0 8px;padding-left:20px}.tcc li{margin:2px 0}
.tcc .barra{display:inline-block;height:9px;border-radius:4px;background:#2a78d6;vertical-align:middle;margin-right:6px}
.tcc .tecnico{font-size:12px;color:#55606b;border-top:1px solid #e3e8ec;padding-top:8px;margin-top:14px}
</style>
"""

pct = lambda x: f"{x:.0%}"
num = lambda x, d=1: f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
rs = lambda x, d=1: ("−" if x < 0 else "") + "R$ " + num(abs(x), d)
de100 = lambda x: f"{round(x * 100):.0f} em cada 100"


def _mostrar(html):
    display(HTML(CSS + f'<div class="tcc">{html}</div>'))


# =============================================================== H1
def mostrar_h1(p_evento=0.40, queda_min=0.20, queda_max=0.50, ruido=0.15, limiar=0.85,
               janela=4, consecutivas=2):
    r = sim.rodar_h1(p_evento=p_evento, queda_min=queda_min, queda_max=queda_max, ruido=ruido,
                     limiar=limiar, janela=int(janela), consecutivas=int(consecutivas))
    t, k = r["trimestral"], r["continuo"]
    preservado = t["perda_mi"] - k["perda_mi"]
    vezes = t["atraso_mediano"] / k["atraso_mediano"] if k["atraso_mediano"] else 0

    if r["suportada"]:
        cls, veredito = "ok", "Sim. A plataforma percebe os problemas bem mais cedo, sem gerar alarmes demais."
        acao = "A regra de alerta está bem calibrada para esta carteira e pode ser usada como está."
    elif r["reducao"] >= 0.5:
        cls, veredito = "parcial", ("Em parte. A plataforma percebe os problemas bem mais cedo, mas avisa "
                                     "sem necessidade com frequência demais.")
        acao = ("Ajuste a regra para ficar menos sensível: suba o número de semanas seguidas abaixo da meta "
                "para 3, ou baixe a meta de alerta para 80%. Nos testes do TCC, isso reduziu os alarmes "
                "falsos para menos de 10 em cada 100 sem perder quase nada da rapidez.")
    else:
        cls, veredito = "nao", "Não. Com estes parâmetros, a plataforma não percebe os problemas cedo o bastante."
        acao = "Revise a regra de alerta: uma meta de alerta mais alta ou uma janela menor aumentam a sensibilidade."

    html = f"""
    <h3>H1 · Quando um projeto de IA começa a falhar, quem percebe primeiro?</h3>
    <p class="pergunta">Uma instituição aprova um projeto de IA prometendo um ganho, por exemplo R$ 20 mil por semana
    em cobrança recuperada. Meses depois, a equipe para de usar a ferramenta ou os dados param de ser atualizados, e o
    ganho cai. <b>Em quanto tempo alguém percebe, e quanto dinheiro se perde até lá?</b></p>

    <h4>O que simulamos</h4>
    <ul>
      <li>Uma carteira fictícia de {r['n']} projetos de IA, acompanhados semana a semana durante um ano.</li>
      <li>{r['com_queda']} deles ({pct(r['com_queda'] / r['n'])}) passam a entregar entre {pct(queda_min)} e
          {pct(queda_max)} menos do que prometeram, a partir de uma semana qualquer do ano.</li>
      <li>Comparamos dois jeitos de acompanhar a mesma carteira:
        <b>reunião trimestral</b>, que olha o resultado do trimestre a cada três meses, como a maioria das empresas faz hoje;
        e <b>a plataforma proposta</b>, que confere toda semana e avisa quando o projeto fica abaixo de
        {pct(limiar)} do prometido por {int(consecutivas)} semana(s) seguida(s).</li>
    </ul>

    <h4>O que encontramos</h4>
    <div class="grade">
      <div class="kpi"><b>{num(k['atraso_mediano'], 0)} semanas</b><span>é o tempo típico para a plataforma avisar que um projeto começou a falhar</span></div>
      <div class="kpi"><b>{num(t['atraso_mediano'], 0)} semanas</b><span>é o tempo típico para a reunião trimestral perceber o mesmo problema</span></div>
      <div class="kpi"><b>{rs(preservado)} mi</b><span>de ganho preservado na carteira por perceber antes (perda de {rs(t['perda_mi'])} mi cai para {rs(k['perda_mi'])} mi)</span></div>
      <div class="kpi"><b>{de100(k['falso_alarme'])}</b><span>projetos que estavam bem receberam um alerta sem motivo (na reunião trimestral: {de100(t['falso_alarme'])})</span></div>
    </div>

    <div class="resposta {cls}"><b>A plataforma resolve o problema? {veredito}</b>
    Ela percebe os problemas cerca de {num(vezes, 0)} vezes mais rápido. A meta definida no TCC era perceber pelo menos
    duas vezes mais rápido com no máximo 10 alarmes falsos em cada 100 projetos saudáveis.</div>
    <p><b>O que fazer com isso:</b> {acao}</p>

    <div class="tecnico">Para a banca: redução de {pct(r['reducao'])} no atraso mediano; falso alarme de
    {num(k['falso_alarme'] * 100)}%; teste de Mann-Whitney unilateral, p {'&lt; 0,001' if r['p'] < 0.001 else '= ' + num(r['p'], 3)}.
    Taxa de detecção: {num(k['taxa_deteccao'] * 100)}% na plataforma e {num(t['taxa_deteccao'] * 100)}% na revisão trimestral.</div>
    """
    _mostrar(html)

    b = r["hist"]["bordas"][:-1]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.bar(b, r["hist"]["trimestral"], width=0.9, align="edge", color=LARANJA, alpha=0.85,
           label="Reunião trimestral")
    ax.bar(b, r["hist"]["continuo"], width=0.9, align="edge", color=AZUL, alpha=0.85,
           label="Plataforma proposta (acompanhamento semanal)")
    ax.set_title("Quantas semanas cada projeto ficou entregando menos até alguém perceber",
                 loc="left", fontsize=11, color=TINTA)
    ax.set_xlabel("Semanas até o alerta (quanto mais à esquerda, melhor)")
    ax.set_ylabel("Número de projetos")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


# =============================================================== H2 e H3
TRADUZ = {"Ritmo de aquisição de clientes": "Vender no ritmo planejado",
          "Preço praticado": "Conseguir cobrar o preço planejado",
          "Custos fixos": "Manter a equipe e os custos fixos no orçamento",
          "Rotatividade de clientes": "Não perder clientes ao longo do tempo",
          "Custo variável de IA e voz": "Custo de uso da IA, voz e nuvem",
          "Taxa de aceite dos agentes": "Taxa de aprovação dos agentes entregues"}


def mostrar_h2(faixas=None, n=10000, meta=0.70):
    r = sim.rodar_h2(faixas=faixas, n=int(n), meta=meta)
    tor = r["tornado"]
    soma = sum(abs(t["rho"]) for t in tor) or 1
    principal = tor[0]
    segundo = tor[1]
    razao = abs(principal["rho"]) / abs(segundo["rho"]) if segundo["rho"] else 0

    if r["h2_suportada"]:
        cls2, ver2 = "ok", f"Sim. A empresa cria valor na grande maioria dos futuros simulados."
    elif r["p_pos"] >= 0.4:
        cls2, ver2 = "parcial", "Talvez. A empresa cria valor em boa parte dos futuros, mas abaixo da meta de segurança."
    else:
        cls2, ver2 = "nao", "Ainda não. Na maioria dos futuros simulados, a empresa não recupera o dinheiro investido."

    linhas = ""
    for t in tor:
        peso = abs(t["rho"]) / soma
        linhas += (f"<tr><td>{TRADUZ.get(t['nome'], t['nome'])}</td>"
                   f"<td><span class='barra' style='width:{max(4, peso * 140):.0f}px'></span>{pct(peso)}</td>"
                   f"<td>{rs(t['pior'])} mi</td><td>{rs(t['melhor'])} mi</td></tr>")

    cond = ""
    if r["p_pos_ritmo_pleno"] is not None:
        cond = (f"Se as entrevistas confirmarem que é possível vender pelo menos no ritmo planejado, a chance de a "
                f"empresa criar valor sobe de {de100(r['p_pos'])} para {de100(r['p_pos_ritmo_pleno'])}.")

    html = f"""
    <h3>H2 e H3 · Vale a pena abrir a empresa que oferece a plataforma?</h3>
    <p class="pergunta">No plano de negócios, com tudo saindo como previsto, a empresa vale hoje
    <b>{rs(r['vpl_base'])} mi</b>, passa a dar lucro no ano {r['base']['equilibrio']} e precisa de
    {rs(r['base']['caixa'])} mi de investimento até lá. <b>Mas e se as coisas não saírem exatamente como previsto?</b></p>

    <h4>O que simulamos</h4>
    <ul>
      <li>Imaginamos {num(r['n'], 0)} futuros possíveis para os próximos cinco anos.</li>
      <li>Em cada futuro, vendas, preço, custos, aprovação dos agentes e perda de clientes variam dentro das faixas
          escolhidas nos controles acima, que vão do cenário ruim ao cenário bom.</li>
      <li>Para cada futuro, calculamos quanto a empresa vale hoje, considerando todo o dinheiro que ela gera ou consome
          (o valor presente líquido, VPL). Valor positivo quer dizer que o negócio paga o investimento com o retorno
          esperado por um investidor de risco, de 25% ao ano.</li>
    </ul>

    <h4>O que encontramos</h4>
    <div class="grade">
      <div class="kpi"><b>{de100(r['p_pos'])}</b><span>futuros a empresa cria valor (meta do TCC: {de100(r['meta'])})</span></div>
      <div class="kpi"><b>{rs(r['p50'])} mi</b><span>é quanto a empresa vale no futuro típico, o do meio da lista{" (negativo: destrói valor)" if r["p50"] < 0 else ""}</span></div>
      <div class="kpi"><b>{rs(r['caixa_p50'])} mi</b><span>de investimento necessário no futuro típico (no plano: {rs(r['base']['caixa'])} mi)</span></div>
      <div class="kpi"><b>{de100(r['equilibrio_5a'])}</b><span>futuros a empresa passa a dar lucro em até cinco anos</span></div>
    </div>
    <div class="resposta {cls2}"><b>H2 · O negócio é viável? {ver2}</b>
    O valor do plano de negócios só aparece quando quase tudo sai como previsto. {cond}</div>

    <h4>H3 · O que mais pesa no resultado</h4>
    <table><tr><th>Fator</th><th>Peso no resultado</th><th>Valor da empresa se der errado</th><th>Valor da empresa se der certo</th></tr>{linhas}</table>
    <div class="resposta {'ok' if r['h3_suportada'] else 'nao'}"><b>O fator decisivo é: {TRADUZ.get(principal['nome'], principal['nome']).lower()}.</b>
    Ele pesa cerca de {num(razao, 0)} vezes mais que o segundo fator. Se der errado, a empresa vale {rs(principal['pior'])} mi;
    se der certo, {rs(principal['melhor'])} mi.</div>
    <p><b>O que fazer com isso:</b> antes de investir, confirmar com 8 a 10 instituições quantas fechariam contrato e em
    quanto tempo. Esse é o número que decide o negócio; preço e custos vêm depois.</p>

    <div class="tecnico">Para a banca: P(VPL &gt; 0) = {num(r['p_pos'] * 100)}% (IC 95%: {num(r['ic'][0] * 100)}% a
    {num(r['ic'][1] * 100)}%); VPL P10 = {num(r['p10'])}, P50 = {num(r['p50'])}, P90 = {num(r['p90'])} (R$ mi).
    O peso de cada fator é a participação de sua correlação de Spearman com o VPL; os valores “se der errado” e
    “se der certo” vêm da análise de tornado, variando um fator por vez.</div>
    """
    _mostrar(html)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.8, 4.1), gridspec_kw={"width_ratios": [1.1, 1]})
    bd = r["hist"]["bordas"]
    larg = bd[1] - bd[0]
    a1.bar(bd[:-1], r["hist"]["negativo"], width=larg, align="edge", color=LARANJA, edgecolor="white",
           linewidth=0.5, label="Não recupera o investimento")
    a1.bar(bd[:-1], r["hist"]["positivo"], width=larg, align="edge", color=AZUL, edgecolor="white",
           linewidth=0.5, label="Cria valor")
    a1.axvline(0, color=TINTA, linewidth=1)
    a1.set_title(f"Quanto a empresa vale hoje em cada um dos {num(r['n'], 0)} futuros", loc="left",
                 fontsize=11, color=TINTA)
    a1.set_xlabel("Valor da empresa hoje (R$ milhões)")
    a1.set_ylabel("Número de futuros")
    a1.spines[["top", "right"]].set_visible(False)
    a1.legend(frameon=False, fontsize=9)

    inv = list(reversed(tor))
    for i, t in enumerate(inv):
        a2.barh(i, t["pior"] - r["vpl_base"], left=r["vpl_base"], color=LARANJA, height=0.6)
        a2.barh(i, t["melhor"] - r["vpl_base"], left=r["vpl_base"], color=AZUL, height=0.6)
    a2.axvline(r["vpl_base"], color=TINTA, linewidth=1)
    a2.set_yticks(range(len(inv)), [TRADUZ.get(t["nome"], t["nome"]) for t in inv], fontsize=9)
    a2.set_title("O que mais muda o valor da empresa", loc="left", fontsize=11, color=TINTA)
    a2.set_xlabel("Valor da empresa hoje (R$ milhões)")
    a2.spines[["top", "right"]].set_visible(False)
    a2.legend(handles=[Patch(color=LARANJA, label="Se der errado"), Patch(color=AZUL, label="Se der certo")],
              frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.3, -0.2), ncol=2)
    fig.tight_layout()
    plt.show()
''',
}
for nome, texto in codigo.items():
    open(nome, 'w', encoding='utf-8').write(texto)

import importlib, simulador, relatorio
importlib.reload(simulador); importlib.reload(relatorio)
print('Simulador pronto. Veja os resultados nas células 2 e 3.')


In [ ]:
#@title 2. H1 · Quando um projeto de IA começa a falhar, quem percebe primeiro? { run: "auto", display-mode: "form" }
#@markdown ### A carteira de projetos simulada
#@markdown Parcela dos projetos que, em algum momento do ano, passa a entregar menos do que prometeu:
parcela_de_projetos_que_falham = 0.40 #@param {type:"slider", min:0.10, max:0.70, step:0.05}
#@markdown Quanto o ganho cai quando o projeto falha (menor e maior queda, em fração do prometido):
menor_queda_do_ganho = 0.20 #@param {type:"slider", min:0.05, max:0.80, step:0.05}
maior_queda_do_ganho = 0.50 #@param {type:"slider", min:0.05, max:0.80, step:0.05}
#@markdown Oscilação normal do resultado de uma semana para outra, mesmo sem problema:
oscilacao_semanal = 0.15 #@param {type:"slider", min:0.05, max:0.30, step:0.01}
#@markdown ### A regra de alerta da plataforma
#@markdown A plataforma avisa quando o projeto fica abaixo desta fração do ganho prometido...
alertar_abaixo_de = 0.85 #@param {type:"slider", min:0.70, max:0.95, step:0.01}
#@markdown ...por este número de semanas seguidas (mais semanas = menos alarmes falsos, alerta um pouco mais tarde):
semanas_seguidas_abaixo = 2 #@param {type:"slider", min:1, max:4, step:1}
#@markdown Número de semanas usadas para calcular a média do resultado (suaviza a oscilação semanal):
semanas_na_media = 4 #@param {type:"slider", min:2, max:8, step:1}

relatorio.mostrar_h1(p_evento=parcela_de_projetos_que_falham,
                     queda_min=min(menor_queda_do_ganho, maior_queda_do_ganho),
                     queda_max=max(menor_queda_do_ganho, maior_queda_do_ganho),
                     ruido=oscilacao_semanal, limiar=alertar_abaixo_de,
                     janela=semanas_na_media, consecutivas=semanas_seguidas_abaixo)


In [ ]:
#@title 3. H2 e H3 · Vale a pena abrir a empresa que oferece a plataforma? { run: "auto", display-mode: "form" }
#@markdown Para cada fator, escolha o **cenário ruim**, o **mais provável** e o **cenário bom**. Valores em relação ao plano de negócios: 1,00 = como planejado; 0,50 = metade; 1,20 = 20% a mais.
#@markdown ### Vendas: novos clientes por ano em relação ao plano (6, 8, 8, 10 e 10)
vendas_cenario_ruim = 0.50 #@param {type:"slider", min:0.2, max:1.6, step:0.05}
vendas_mais_provavel = 1.00 #@param {type:"slider", min:0.2, max:1.6, step:0.05}
vendas_cenario_bom = 1.20 #@param {type:"slider", min:0.2, max:1.6, step:0.05}
#@markdown ### Preço cobrado em relação à tabela de preços
preco_cenario_ruim = 0.80 #@param {type:"slider", min:0.5, max:1.5, step:0.05}
preco_mais_provavel = 1.00 #@param {type:"slider", min:0.5, max:1.5, step:0.05}
preco_cenario_bom = 1.10 #@param {type:"slider", min:0.5, max:1.5, step:0.05}
#@markdown ### Agentes entregues que o cliente aprova (e paga)
aprovacao_cenario_ruim = 0.60 #@param {type:"slider", min:0.3, max:1.0, step:0.05}
aprovacao_cenario_bom = 0.90 #@param {type:"slider", min:0.3, max:1.0, step:0.05}
#@markdown ### Custo de uso da IA, voz e nuvem, em fração da receita (o plano usa 0,06 = 6%)
custo_ia_cenario_bom = 0.04 #@param {type:"slider", min:0.0, max:0.25, step:0.01}
custo_ia_mais_provavel = 0.06 #@param {type:"slider", min:0.0, max:0.25, step:0.01}
custo_ia_cenario_ruim = 0.12 #@param {type:"slider", min:0.0, max:0.25, step:0.01}
#@markdown ### Equipe e custos fixos em relação ao orçamento (R$ 2,2 a 6,4 milhões por ano)
fixos_cenario_bom = 0.90 #@param {type:"slider", min:0.6, max:1.5, step:0.05}
fixos_mais_provavel = 1.00 #@param {type:"slider", min:0.6, max:1.5, step:0.05}
fixos_cenario_ruim = 1.15 #@param {type:"slider", min:0.6, max:1.5, step:0.05}
#@markdown ### Clientes perdidos por ano (0,05 = 5 em cada 100 clientes cancelam)
perda_clientes_cenario_bom = 0.00 #@param {type:"slider", min:0.0, max:0.4, step:0.01}
perda_clientes_mais_provavel = 0.05 #@param {type:"slider", min:0.0, max:0.4, step:0.01}
perda_clientes_cenario_ruim = 0.20 #@param {type:"slider", min:0.0, max:0.4, step:0.01}
#@markdown ### Simulação
futuros_simulados = 10000 #@param [1000, 10000, 50000] {type:"raw"}
#@markdown Meta: em quantos futuros, em cada 100, a empresa precisa criar valor para o negócio ser considerado viável:
meta_de_seguranca = 0.70 #@param {type:"slider", min:0.3, max:0.95, step:0.05}

def faixa(a, m, b):
    a, b = min(a, b), max(a, b)
    return (a, None if m is None else min(max(m, a), b), b)

relatorio.mostrar_h2(faixas={
    "mult_clientes": faixa(vendas_cenario_ruim, vendas_mais_provavel, vendas_cenario_bom),
    "mult_preco": faixa(preco_cenario_ruim, preco_mais_provavel, preco_cenario_bom),
    "aceite": faixa(aprovacao_cenario_ruim, None, aprovacao_cenario_bom),
    "custo_var": faixa(custo_ia_cenario_bom, custo_ia_mais_provavel, custo_ia_cenario_ruim),
    "mult_fixos": faixa(fixos_cenario_bom, fixos_mais_provavel, fixos_cenario_ruim),
    "churn": faixa(perda_clientes_cenario_bom, perda_clientes_mais_provavel, perda_clientes_cenario_ruim),
}, n=futuros_simulados, meta=meta_de_seguranca)
